# v2 — 04: Modeling
**Iteration 2** model — unified dynasty value predictor including rookies.

New features vs v1:
- `is_rookie` flag
- Injury history: `games_missed_prev1`, `career_games_missed_rate`, `ir_flag_prev1`
- Draft capital: `draft_pick`, `draft_round`, `age_at_draft`
- Combine athleticism: `combine_forty`, `combine_weight`, `combine_vertical`
- College production: `college_rec_yards`, `college_rec_tds`, `college_dominator_rate`

**Prerequisite:** Run `v2_03_pipeline_cleaning.ipynb` to build the updated `model_ready` table.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.predictor import (
    load_model_data, train_and_evaluate, get_feature_importance, FEATURE_COLS
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)
%matplotlib inline

## 1. Load Model Data

In [ ]:
df = load_model_data()
print(f'Shape: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
if 'is_rookie' in df.columns:
    print(f'Rookies: {int(df["is_rookie"].sum())} | Veterans: {int((df["is_rookie"] == 0).sum())}')
print(f'Positions: {df["position"].value_counts().to_dict()}')

## 2. Feature Availability

In [ ]:
available = [c for c in FEATURE_COLS if c in df.columns]
missing = [c for c in FEATURE_COLS if c not in df.columns]
print(f'{len(available)} of {len(FEATURE_COLS)} features available')
if missing:
    print(f'Missing: {missing}')
print('\nNull rates by feature:')
for f in available:
    pct = df[f].isna().mean() * 100
    if pct > 0:
        print(f'  {f:35s}: {pct:.1f}% null (handled by imputer)')

## 3. Train & Evaluate All Models

In [ ]:
results, pred_df = train_and_evaluate(df)

## 4. Results Comparison

In [ ]:
results_df = pd.DataFrame(results['results']).sort_values('mae')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['mae', 'rmse', 'r2']):
    colors = ['darkorange' if m == results['best_model'] else 'steelblue'
              for m in results_df['model']]
    ax.barh(results_df['model'], results_df[metric], color=colors)
    ax.set_title(metric.upper())
    if metric in ['mae', 'rmse']:
        ax.invert_xaxis()
plt.suptitle(f'v2 Model Comparison — Best: {results["best_model"]}', y=1.02)
plt.tight_layout()
plt.show()
results_df

## 5. Feature Importance

In [ ]:
feat_df = get_feature_importance(df)

fig, ax = plt.subplots(figsize=(10, 9))
top = feat_df.head(20)
ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('v2 Random Forest — Top 20 Feature Importances')
plt.tight_layout()
plt.show()

print('\nTop 20 features:')
print(feat_df.head(20).to_string(index=False))

## 6. Rookie-Specific Analysis
How well does the model predict rookie year output?

In [ ]:
if 'is_rookie' in pred_df.columns:
    from sklearn.metrics import mean_absolute_error
    rookies_pred = pred_df[pred_df['is_rookie'] == 1].dropna(subset=['actual_ppr_next', 'predicted_ppr_next'])
    vets_pred    = pred_df[pred_df['is_rookie'] == 0].dropna(subset=['actual_ppr_next', 'predicted_ppr_next'])

    if len(rookies_pred) > 0:
        rmae = mean_absolute_error(rookies_pred['actual_ppr_next'], rookies_pred['predicted_ppr_next'])
        vmae = mean_absolute_error(vets_pred['actual_ppr_next'], vets_pred['predicted_ppr_next'])
        print(f'Rookie MAE: {rmae:.2f} | Veteran MAE: {vmae:.2f}')
        print(f'Rookies in test set: {len(rookies_pred)}')

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(rookies_pred['actual_ppr_next'], rookies_pred['predicted_ppr_next'],
                   alpha=0.5, s=30, label='Rookies')
        lim = max(rookies_pred['actual_ppr_next'].max(), rookies_pred['predicted_ppr_next'].max()) * 1.05
        ax.plot([0, lim], [0, lim], 'r--', alpha=0.5, label='Perfect')
        ax.set_title('Rookie Predictions — Actual vs Predicted')
        ax.set_xlabel('Actual PPR Points')
        ax.set_ylabel('Predicted PPR Points')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 7. Dynasty Rankings Output (2024 prediction)

In [ ]:
display_cols = ['predicted_rank', 'full_name', 'position', 'predicted_ppr_next',
                'actual_ppr_next', 'error']
if 'is_rookie' in pred_df.columns:
    display_cols.insert(3, 'is_rookie')
available_display = [c for c in display_cols if c in pred_df.columns]

print('=== Top 30 Overall Dynasty Rankings ===')
pred_df[available_display].head(30)

In [ ]:
# Top rookies specifically
if 'is_rookie' in pred_df.columns:
    rookie_rankings = pred_df[pred_df['is_rookie'] == 1].head(15)
    print('=== Top 15 Rookie Dynasty Rankings ===')
    print(rookie_rankings[available_display].to_string(index=False))